# Writer-Disjoint Verification Protocol

This notebook defines reproducible writer-verification protocols for QUWI.

## Purpose

- Use development-training writers only for model development
- Keep validation and official-test writers completely unseen
- Construct genuine pairs from handwriting samples of the same writer
- Construct condition-matched impostor pairs from different writers
- Separate within-script and cross-script verification conditions
- Track variable-text and same-text pair configurations
- Generate deterministic validation and official-test pair manifests
- Verify that no writer leakage exists across experiment splits

The generated protocols will support later evaluation using similarity scores, ROC curves, equal error rate, and operating-point metrics.

In [1]:
import json
from itertools import combinations, product
from pathlib import Path

import numpy as np
import pandas as pd

In [2]:
PROJECT_ROOT = Path.cwd().resolve()

while PROJECT_ROOT != PROJECT_ROOT.parent and not (
    PROJECT_ROOT / "pyproject.toml"
).exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

assert (PROJECT_ROOT / "pyproject.toml").exists()

SPLIT_PATH = (
    PROJECT_ROOT
    / "splits"
    / "quwi_writer_disjoint_split_seed42.csv"
)

PAIR_DIR = (
    PROJECT_ROOT
    / "splits"
    / "verification"
)

REPORT_DIR = (
    PROJECT_ROOT
    / "reports"
    / "verification_protocol"
)

PAIR_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

split_df = pd.read_csv(SPLIT_PATH)

required_columns = {
    "filename",
    "writer",
    "page_id",
    "language",
    "same_text",
    "experiment_split",
}

missing_columns = (
    required_columns
    - set(split_df.columns)
)

assert not missing_columns

split_summary_df = (
    split_df
    .groupby(
        "experiment_split",
        as_index=False,
    )
    .agg(
        images=("filename", "size"),
        writers=("writer", "nunique"),
    )
)

print("Project root:", PROJECT_ROOT)
print("Split file exists:", SPLIT_PATH.exists())
print("Total images:", len(split_df))
print("Total writers:", split_df["writer"].nunique())
print("Pair directory:", PAIR_DIR)
print("Report directory:", REPORT_DIR)

display(split_summary_df)

assert len(split_df) == 1900
assert split_df["writer"].nunique() == 475

Project root: /home/arijit/Documents/handwriting-cross-script-research
Split file exists: True
Total images: 1900
Total writers: 475
Pair directory: /home/arijit/Documents/handwriting-cross-script-research/splits/verification
Report directory: /home/arijit/Documents/handwriting-cross-script-research/reports/verification_protocol


,experiment_split,images,writers
0,development_train,904,226
1,official_test,772,193
2,validation,224,56


In [3]:
page_metadata_df = pd.DataFrame(
    [
        {
            "page_id": 1,
            "language": "Arabic",
            "same_text": 0,
            "text_condition": "variable",
        },
        {
            "page_id": 2,
            "language": "Arabic",
            "same_text": 1,
            "text_condition": "same",
        },
        {
            "page_id": 3,
            "language": "English",
            "same_text": 0,
            "text_condition": "variable",
        },
        {
            "page_id": 4,
            "language": "English",
            "same_text": 1,
            "text_condition": "same",
        },
    ]
)

pair_conditions_df = pd.DataFrame(
    [
        {
            "condition_id": "arabic_variable_same",
            "page_a": 1,
            "page_b": 2,
            "script_relation": "within_arabic",
            "text_relation": "variable_same",
        },
        {
            "condition_id": "english_variable_same",
            "page_a": 3,
            "page_b": 4,
            "script_relation": "within_english",
            "text_relation": "variable_same",
        },
        {
            "condition_id": "cross_variable_variable",
            "page_a": 1,
            "page_b": 3,
            "script_relation": "cross_script",
            "text_relation": "variable_variable",
        },
        {
            "condition_id": "cross_variable_same",
            "page_a": 1,
            "page_b": 4,
            "script_relation": "cross_script",
            "text_relation": "variable_same",
        },
        {
            "condition_id": "cross_same_variable",
            "page_a": 2,
            "page_b": 3,
            "script_relation": "cross_script",
            "text_relation": "same_variable",
        },
        {
            "condition_id": "cross_same_same",
            "page_a": 2,
            "page_b": 4,
            "script_relation": "cross_script",
            "text_relation": "same_same",
        },
    ]
)

page_lookup = (
    page_metadata_df
    .set_index("page_id")
    .to_dict(orient="index")
)

pair_conditions_df["language_a"] = (
    pair_conditions_df["page_a"]
    .map(
        lambda page_id: page_lookup[
            page_id
        ]["language"]
    )
)

pair_conditions_df["language_b"] = (
    pair_conditions_df["page_b"]
    .map(
        lambda page_id: page_lookup[
            page_id
        ]["language"]
    )
)

pair_conditions_df["text_condition_a"] = (
    pair_conditions_df["page_a"]
    .map(
        lambda page_id: page_lookup[
            page_id
        ]["text_condition"]
    )
)

pair_conditions_df["text_condition_b"] = (
    pair_conditions_df["page_b"]
    .map(
        lambda page_id: page_lookup[
            page_id
        ]["text_condition"]
    )
)

display(page_metadata_df)
display(pair_conditions_df)

print(
    "Verification conditions:",
    len(pair_conditions_df),
)

print(
    "Within-script conditions:",
    (
        pair_conditions_df[
            "script_relation"
        ] != "cross_script"
    ).sum(),
)

print(
    "Cross-script conditions:",
    (
        pair_conditions_df[
            "script_relation"
        ] == "cross_script"
    ).sum(),
)

assert len(pair_conditions_df) == 6
assert (
    pair_conditions_df[
        ["page_a", "page_b"]
    ]
    .drop_duplicates()
    .shape[0]
    == 6
)

,page_id,language,same_text,text_condition
0,1,Arabic,0,variable
1,2,Arabic,1,same
2,3,English,0,variable
3,4,English,1,same


,condition_id,page_a,page_b,script_relation,text_relation,language_a,language_b,text_condition_a,text_condition_b
0,arabic_variable_same,1,2,within_arabic,variable_same,Arabic,Arabic,variable,same
1,english_variable_same,3,4,within_english,variable_same,English,English,variable,same
2,cross_variable_variable,1,3,cross_script,variable_variable,Arabic,English,variable,variable
3,cross_variable_same,1,4,cross_script,variable_same,Arabic,English,variable,same
4,cross_same_variable,2,3,cross_script,same_variable,Arabic,English,same,variable
5,cross_same_same,2,4,cross_script,same_same,Arabic,English,same,same


Verification conditions: 6
Within-script conditions: 2
Cross-script conditions: 4


In [4]:
evaluation_splits = {
    "validation": split_df[
        split_df["experiment_split"] == "validation"
    ].copy(),
    "official_test": split_df[
        split_df["experiment_split"] == "official_test"
    ].copy(),
}

page_audit_rows = []

for split_name, metadata in evaluation_splits.items():
    writer_page_counts = (
        metadata
        .groupby(["writer", "page_id"])
        .size()
    )

    writer_image_counts = (
        metadata
        .groupby("writer")
        .size()
    )

    writer_unique_pages = (
        metadata
        .groupby("writer")["page_id"]
        .nunique()
    )

    page_audit_rows.append(
        {
            "experiment_split": split_name,
            "images": len(metadata),
            "writers": metadata["writer"].nunique(),
            "minimum_images_per_writer": writer_image_counts.min(),
            "maximum_images_per_writer": writer_image_counts.max(),
            "minimum_unique_pages": writer_unique_pages.min(),
            "maximum_unique_pages": writer_unique_pages.max(),
            "duplicate_writer_page_entries": int(
                (writer_page_counts > 1).sum()
            ),
        }
    )

page_audit_df = pd.DataFrame(page_audit_rows)

display(page_audit_df)

validation_writers = set(
    evaluation_splits["validation"]["writer"]
)

test_writers = set(
    evaluation_splits["official_test"]["writer"]
)

development_writers = set(
    split_df.loc[
        split_df["experiment_split"] == "development_train",
        "writer",
    ]
)

print(
    "Validation-test writer overlap:",
    len(validation_writers & test_writers),
)

print(
    "Development-validation writer overlap:",
    len(development_writers & validation_writers),
)

print(
    "Development-test writer overlap:",
    len(development_writers & test_writers),
)

assert page_audit_df[
    "minimum_images_per_writer"
].eq(4).all()

assert page_audit_df[
    "maximum_images_per_writer"
].eq(4).all()

assert page_audit_df[
    "minimum_unique_pages"
].eq(4).all()

assert page_audit_df[
    "maximum_unique_pages"
].eq(4).all()

assert page_audit_df[
    "duplicate_writer_page_entries"
].eq(0).all()

assert not validation_writers & test_writers
assert not development_writers & validation_writers
assert not development_writers & test_writers

,experiment_split,images,writers,minimum_images_per_writer,maximum_images_per_writer,minimum_unique_pages,maximum_unique_pages,duplicate_writer_page_entries
0,validation,224,56,4,4,4,4,0
1,official_test,772,193,4,4,4,4,0


Validation-test writer overlap: 0
Development-validation writer overlap: 0
Development-test writer overlap: 0


In [5]:
def build_verification_pairs(
    metadata,
    split_name,
    conditions,
):
    split_metadata = metadata[
        metadata["experiment_split"] == split_name
    ].copy()

    condition_frames = []

    for condition in conditions.itertuples(index=False):
        page_a_df = (
            split_metadata[
                split_metadata["page_id"] == condition.page_a
            ][
                [
                    "filename",
                    "writer",
                    "page_id",
                    "language",
                    "same_text",
                ]
            ]
            .rename(
                columns={
                    "filename": "filename_a",
                    "writer": "writer_a",
                    "page_id": "page_a",
                    "language": "language_a",
                    "same_text": "same_text_a",
                }
            )
            .sort_values("writer_a")
            .reset_index(drop=True)
        )

        page_b_df = (
            split_metadata[
                split_metadata["page_id"] == condition.page_b
            ][
                [
                    "filename",
                    "writer",
                    "page_id",
                    "language",
                    "same_text",
                ]
            ]
            .rename(
                columns={
                    "filename": "filename_b",
                    "writer": "writer_b",
                    "page_id": "page_b",
                    "language": "language_b",
                    "same_text": "same_text_b",
                }
            )
            .sort_values("writer_b")
            .reset_index(drop=True)
        )

        pairs = page_a_df.merge(
            page_b_df,
            how="cross",
        )

        pairs["experiment_split"] = split_name
        pairs["condition_id"] = condition.condition_id
        pairs["script_relation"] = condition.script_relation
        pairs["text_relation"] = condition.text_relation

        pairs["pair_label"] = (
            pairs["writer_a"] == pairs["writer_b"]
        ).astype(int)

        pairs["pair_type"] = np.where(
            pairs["pair_label"] == 1,
            "genuine",
            "impostor",
        )

        condition_frames.append(pairs)

    pair_df = pd.concat(
        condition_frames,
        ignore_index=True,
    )

    pair_df.insert(
        0,
        "pair_id",
        [
            f"{split_name}_{index:07d}"
            for index in range(1, len(pair_df) + 1)
        ],
    )

    return pair_df


validation_pairs_df = build_verification_pairs(
    split_df,
    "validation",
    pair_conditions_df,
)

official_test_pairs_df = build_verification_pairs(
    split_df,
    "official_test",
    pair_conditions_df,
)

print(
    "Validation pairs:",
    len(validation_pairs_df),
)

print(
    "Validation genuine:",
    (
        validation_pairs_df["pair_type"] == "genuine"
    ).sum(),
)

print(
    "Validation impostor:",
    (
        validation_pairs_df["pair_type"] == "impostor"
    ).sum(),
)

print(
    "Official-test pairs:",
    len(official_test_pairs_df),
)

print(
    "Official-test genuine:",
    (
        official_test_pairs_df["pair_type"] == "genuine"
    ).sum(),
)

print(
    "Official-test impostor:",
    (
        official_test_pairs_df["pair_type"] == "impostor"
    ).sum(),
)

print(
    "Validation duplicate pairs:",
    validation_pairs_df.duplicated(
        subset=[
            "condition_id",
            "filename_a",
            "filename_b",
        ]
    ).sum(),
)

print(
    "Official-test duplicate pairs:",
    official_test_pairs_df.duplicated(
        subset=[
            "condition_id",
            "filename_a",
            "filename_b",
        ]
    ).sum(),
)

assert len(validation_pairs_df) == 18_816
assert len(official_test_pairs_df) == 223_494

assert (
    validation_pairs_df["pair_type"] == "genuine"
).sum() == 336

assert (
    validation_pairs_df["pair_type"] == "impostor"
).sum() == 18_480

assert (
    official_test_pairs_df["pair_type"] == "genuine"
).sum() == 1_158

assert (
    official_test_pairs_df["pair_type"] == "impostor"
).sum() == 222_336

Validation pairs: 18816
Validation genuine: 336
Validation impostor: 18480
Official-test pairs: 223494
Official-test genuine: 1158
Official-test impostor: 222336
Validation duplicate pairs: 0
Official-test duplicate pairs: 0


In [6]:
def summarize_pair_manifest(
    pair_df,
    expected_writer_count,
):
    rows = []

    for condition_id, group in pair_df.groupby(
        "condition_id",
        sort=False,
    ):
        rows.append(
            {
                "experiment_split": group[
                    "experiment_split"
                ].iloc[0],
                "condition_id": condition_id,
                "script_relation": group[
                    "script_relation"
                ].iloc[0],
                "text_relation": group[
                    "text_relation"
                ].iloc[0],
                "page_a": group["page_a"].iloc[0],
                "page_b": group["page_b"].iloc[0],
                "writers_a": group[
                    "writer_a"
                ].nunique(),
                "writers_b": group[
                    "writer_b"
                ].nunique(),
                "pairs": len(group),
                "genuine_pairs": (
                    group["pair_type"]
                    == "genuine"
                ).sum(),
                "impostor_pairs": (
                    group["pair_type"]
                    == "impostor"
                ).sum(),
            }
        )

    summary = pd.DataFrame(rows)

    assert summary["writers_a"].eq(
        expected_writer_count
    ).all()

    assert summary["writers_b"].eq(
        expected_writer_count
    ).all()

    assert summary["pairs"].eq(
        expected_writer_count**2
    ).all()

    assert summary["genuine_pairs"].eq(
        expected_writer_count
    ).all()

    assert summary["impostor_pairs"].eq(
        expected_writer_count
        * (expected_writer_count - 1)
    ).all()

    return summary


validation_condition_summary_df = summarize_pair_manifest(
    validation_pairs_df,
    56,
)

test_condition_summary_df = summarize_pair_manifest(
    official_test_pairs_df,
    193,
)

condition_summary_df = pd.concat(
    [
        validation_condition_summary_df,
        test_condition_summary_df,
    ],
    ignore_index=True,
)

validation_label_integrity = (
    validation_pairs_df["pair_label"]
    == (
        validation_pairs_df["writer_a"]
        == validation_pairs_df["writer_b"]
    ).astype(int)
).all()

test_label_integrity = (
    official_test_pairs_df["pair_label"]
    == (
        official_test_pairs_df["writer_a"]
        == official_test_pairs_df["writer_b"]
    ).astype(int)
).all()

validation_type_integrity = (
    (
        (
            validation_pairs_df["pair_label"] == 1
        )
        == (
            validation_pairs_df["pair_type"]
            == "genuine"
        )
    ).all()
)

test_type_integrity = (
    (
        (
            official_test_pairs_df["pair_label"] == 1
        )
        == (
            official_test_pairs_df["pair_type"]
            == "genuine"
        )
    ).all()
)

condition_lookup = (
    pair_conditions_df
    .set_index("condition_id")[
        ["page_a", "page_b"]
    ]
    .to_dict(orient="index")
)

for pair_df in [
    validation_pairs_df,
    official_test_pairs_df,
]:
    for condition_id, group in pair_df.groupby(
        "condition_id"
    ):
        expected = condition_lookup[
            condition_id
        ]

        assert group["page_a"].eq(
            expected["page_a"]
        ).all()

        assert group["page_b"].eq(
            expected["page_b"]
        ).all()

display(condition_summary_df)

print(
    "Validation label integrity:",
    validation_label_integrity,
)

print(
    "Official-test label integrity:",
    test_label_integrity,
)

print(
    "Validation pair-type integrity:",
    validation_type_integrity,
)

print(
    "Official-test pair-type integrity:",
    test_type_integrity,
)

assert validation_label_integrity
assert test_label_integrity
assert validation_type_integrity
assert test_type_integrity

,experiment_split,condition_id,script_relation,text_relation,page_a,page_b,writers_a,writers_b,pairs,genuine_pairs,impostor_pairs
0,validation,arabic_variable_same,within_arabic,variable_same,1,2,56,56,3136,56,3080
1,validation,english_variable_same,within_english,variable_same,3,4,56,56,3136,56,3080
2,validation,cross_variable_variable,cross_script,variable_variable,1,3,56,56,3136,56,3080
3,validation,cross_variable_same,cross_script,variable_same,1,4,56,56,3136,56,3080
4,validation,cross_same_variable,cross_script,same_variable,2,3,56,56,3136,56,3080
5,validation,cross_same_same,cross_script,same_same,2,4,56,56,3136,56,3080
6,official_test,arabic_variable_same,within_arabic,variable_same,1,2,193,193,37249,193,37056
7,official_test,english_variable_same,within_english,variable_same,3,4,193,193,37249,193,37056
8,official_test,cross_variable_variable,cross_script,variable_variable,1,3,193,193,37249,193,37056
9,official_test,cross_variable_same,cross_script,variable_same,1,4,193,193,37249,193,37056


Validation label integrity: True
Official-test label integrity: True
Validation pair-type integrity: True
Official-test pair-type integrity: True


In [7]:
VALIDATION_PAIR_PATH = (
    PAIR_DIR
    / "quwi_validation_verification_pairs.csv"
)

TEST_PAIR_PATH = (
    PAIR_DIR
    / "quwi_official_test_verification_pairs.csv"
)

CONDITION_PATH = (
    PAIR_DIR
    / "quwi_verification_conditions.csv"
)

CONDITION_SUMMARY_PATH = (
    REPORT_DIR
    / "pair_condition_summary.csv"
)

PROTOCOL_SUMMARY_PATH = (
    REPORT_DIR
    / "quwi_verification_protocol_summary.json"
)

validation_pairs_df.to_csv(
    VALIDATION_PAIR_PATH,
    index=False,
)

official_test_pairs_df.to_csv(
    TEST_PAIR_PATH,
    index=False,
)

pair_conditions_df.to_csv(
    CONDITION_PATH,
    index=False,
)

condition_summary_df.to_csv(
    CONDITION_SUMMARY_PATH,
    index=False,
)

protocol_summary = {
    "dataset": "QUWI",
    "protocol": "writer_disjoint_exhaustive_verification",
    "development_train_writers": 226,
    "validation_writers": 56,
    "official_test_writers": 193,
    "verification_conditions": len(
        pair_conditions_df
    ),
    "within_script_conditions": 2,
    "cross_script_conditions": 4,
    "validation": {
        "pairs": len(
            validation_pairs_df
        ),
        "genuine_pairs": int(
            (
                validation_pairs_df[
                    "pair_type"
                ]
                == "genuine"
            ).sum()
        ),
        "impostor_pairs": int(
            (
                validation_pairs_df[
                    "pair_type"
                ]
                == "impostor"
            ).sum()
        ),
    },
    "official_test": {
        "pairs": len(
            official_test_pairs_df
        ),
        "genuine_pairs": int(
            (
                official_test_pairs_df[
                    "pair_type"
                ]
                == "genuine"
            ).sum()
        ),
        "impostor_pairs": int(
            (
                official_test_pairs_df[
                    "pair_type"
                ]
                == "impostor"
            ).sum()
        ),
    },
    "writer_overlap": {
        "development_validation": len(
            development_writers
            & validation_writers
        ),
        "development_test": len(
            development_writers
            & test_writers
        ),
        "validation_test": len(
            validation_writers
            & test_writers
        ),
    },
    "pair_label_integrity": {
        "validation": bool(
            validation_label_integrity
        ),
        "official_test": bool(
            test_label_integrity
        ),
    },
}

with PROTOCOL_SUMMARY_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        protocol_summary,
        file,
        indent=2,
    )

saved_paths = [
    VALIDATION_PAIR_PATH,
    TEST_PAIR_PATH,
    CONDITION_PATH,
    CONDITION_SUMMARY_PATH,
    PROTOCOL_SUMMARY_PATH,
]

for path in saved_paths:
    print(
        path.name,
        ":",
        path.exists(),
    )

print(
    "Validation CSV size:",
    round(
        VALIDATION_PAIR_PATH.stat().st_size
        / 1024**2,
        2,
    ),
    "MB",
)

print(
    "Official-test CSV size:",
    round(
        TEST_PAIR_PATH.stat().st_size
        / 1024**2,
        2,
    ),
    "MB",
)

assert all(
    path.exists()
    for path in saved_paths
)

quwi_validation_verification_pairs.csv : True
quwi_official_test_verification_pairs.csv : True
quwi_verification_conditions.csv : True
pair_condition_summary.csv : True
quwi_verification_protocol_summary.json : True
Validation CSV size: 2.53 MB
Official-test CSV size: 31.54 MB


In [8]:
def audit_pair_balance(
    pair_df,
    split_name,
    expected_writer_count,
):
    expected_filenames = set(
        split_df.loc[
            split_df["experiment_split"] == split_name,
            "filename",
        ]
    )

    used_filenames = (
        set(pair_df["filename_a"])
        | set(pair_df["filename_b"])
    )

    unexpected_filenames = (
        used_filenames - expected_filenames
    )

    rows = []

    for condition_id, group in pair_df.groupby(
        "condition_id",
        sort=False,
    ):
        genuine = group[
            group["pair_label"] == 1
        ]

        impostor = group[
            group["pair_label"] == 0
        ]

        impostor_a_counts = (
            impostor["writer_a"]
            .value_counts()
        )

        impostor_b_counts = (
            impostor["writer_b"]
            .value_counts()
        )

        rows.append(
            {
                "experiment_split": split_name,
                "condition_id": condition_id,
                "genuine_writers": genuine[
                    "writer_a"
                ].nunique(),
                "genuine_pairs_per_writer_min": genuine[
                    "writer_a"
                ].value_counts().min(),
                "genuine_pairs_per_writer_max": genuine[
                    "writer_a"
                ].value_counts().max(),
                "impostor_a_per_writer_min": impostor_a_counts.min(),
                "impostor_a_per_writer_max": impostor_a_counts.max(),
                "impostor_b_per_writer_min": impostor_b_counts.min(),
                "impostor_b_per_writer_max": impostor_b_counts.max(),
            }
        )

    audit_df = pd.DataFrame(rows)

    print(
        split_name,
        "used filenames:",
        len(used_filenames),
    )

    print(
        split_name,
        "unexpected filenames:",
        len(unexpected_filenames),
    )

    print(
        split_name,
        "same-file pairs:",
        (
            pair_df["filename_a"]
            == pair_df["filename_b"]
        ).sum(),
    )

    assert used_filenames == expected_filenames
    assert not unexpected_filenames

    assert not (
        pair_df["filename_a"]
        == pair_df["filename_b"]
    ).any()

    assert audit_df[
        "genuine_writers"
    ].eq(expected_writer_count).all()

    assert audit_df[
        "genuine_pairs_per_writer_min"
    ].eq(1).all()

    assert audit_df[
        "genuine_pairs_per_writer_max"
    ].eq(1).all()

    assert audit_df[
        "impostor_a_per_writer_min"
    ].eq(expected_writer_count - 1).all()

    assert audit_df[
        "impostor_a_per_writer_max"
    ].eq(expected_writer_count - 1).all()

    assert audit_df[
        "impostor_b_per_writer_min"
    ].eq(expected_writer_count - 1).all()

    assert audit_df[
        "impostor_b_per_writer_max"
    ].eq(expected_writer_count - 1).all()

    return audit_df


validation_balance_audit_df = audit_pair_balance(
    validation_pairs_df,
    "validation",
    56,
)

test_balance_audit_df = audit_pair_balance(
    official_test_pairs_df,
    "official_test",
    193,
)

pair_balance_audit_df = pd.concat(
    [
        validation_balance_audit_df,
        test_balance_audit_df,
    ],
    ignore_index=True,
)

display(pair_balance_audit_df)

validation used filenames: 224
validation unexpected filenames: 0
validation same-file pairs: 0
official_test used filenames: 772
official_test unexpected filenames: 0
official_test same-file pairs: 0


,experiment_split,condition_id,genuine_writers,genuine_pairs_per_writer_min,genuine_pairs_per_writer_max,impostor_a_per_writer_min,impostor_a_per_writer_max,impostor_b_per_writer_min,impostor_b_per_writer_max
0,validation,arabic_variable_same,56,1,1,55,55,55,55
1,validation,english_variable_same,56,1,1,55,55,55,55
2,validation,cross_variable_variable,56,1,1,55,55,55,55
3,validation,cross_variable_same,56,1,1,55,55,55,55
4,validation,cross_same_variable,56,1,1,55,55,55,55
5,validation,cross_same_same,56,1,1,55,55,55,55
6,official_test,arabic_variable_same,193,1,1,192,192,192,192
7,official_test,english_variable_same,193,1,1,192,192,192,192
8,official_test,cross_variable_variable,193,1,1,192,192,192,192
9,official_test,cross_variable_same,193,1,1,192,192,192,192


In [9]:
PAIR_BALANCE_PATH = (
    REPORT_DIR
    / "pair_balance_audit.csv"
)

WRITER_LEAKAGE_PATH = (
    REPORT_DIR
    / "writer_leakage_audit.json"
)

pair_balance_audit_df.to_csv(
    PAIR_BALANCE_PATH,
    index=False,
)

writer_leakage_audit = {
    "development_train_writers": len(
        development_writers
    ),
    "validation_writers": len(
        validation_writers
    ),
    "official_test_writers": len(
        test_writers
    ),
    "development_validation_overlap": len(
        development_writers
        & validation_writers
    ),
    "development_test_overlap": len(
        development_writers
        & test_writers
    ),
    "validation_test_overlap": len(
        validation_writers
        & test_writers
    ),
    "validation_used_filenames": len(
        set(validation_pairs_df["filename_a"])
        | set(validation_pairs_df["filename_b"])
    ),
    "official_test_used_filenames": len(
        set(official_test_pairs_df["filename_a"])
        | set(official_test_pairs_df["filename_b"])
    ),
    "validation_unexpected_filenames": 0,
    "official_test_unexpected_filenames": 0,
    "validation_same_file_pairs": int(
        (
            validation_pairs_df["filename_a"]
            == validation_pairs_df["filename_b"]
        ).sum()
    ),
    "official_test_same_file_pairs": int(
        (
            official_test_pairs_df["filename_a"]
            == official_test_pairs_df["filename_b"]
        ).sum()
    ),
}

with WRITER_LEAKAGE_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        writer_leakage_audit,
        file,
        indent=2,
    )

print(
    "Pair balance audit:",
    PAIR_BALANCE_PATH.exists(),
)

print(
    "Writer leakage audit:",
    WRITER_LEAKAGE_PATH.exists(),
)

print(
    "Development-validation overlap:",
    writer_leakage_audit[
        "development_validation_overlap"
    ],
)

print(
    "Development-test overlap:",
    writer_leakage_audit[
        "development_test_overlap"
    ],
)

print(
    "Validation-test overlap:",
    writer_leakage_audit[
        "validation_test_overlap"
    ],
)

print(
    "Validation filenames covered:",
    writer_leakage_audit[
        "validation_used_filenames"
    ],
)

print(
    "Official-test filenames covered:",
    writer_leakage_audit[
        "official_test_used_filenames"
    ],
)

Pair balance audit: True
Writer leakage audit: True
Development-validation overlap: 0
Development-test overlap: 0
Validation-test overlap: 0
Validation filenames covered: 224
Official-test filenames covered: 772


In [10]:
PAGE_AUDIT_PATH = (
    REPORT_DIR
    / "writer_page_audit.csv"
)

FINAL_SUMMARY_PATH = (
    REPORT_DIR
    / "verification_protocol_final_summary.json"
)

page_audit_df.to_csv(
    PAGE_AUDIT_PATH,
    index=False,
)


def make_json_safe(value):
    if isinstance(value, dict):
        return {
            key: make_json_safe(item)
            for key, item in value.items()
        }

    if isinstance(value, list):
        return [
            make_json_safe(item)
            for item in value
        ]

    if isinstance(value, tuple):
        return [
            make_json_safe(item)
            for item in value
        ]

    if isinstance(value, np.integer):
        return int(value)

    if isinstance(value, np.floating):
        return float(value)

    if isinstance(value, np.bool_):
        return bool(value)

    return value


final_protocol_summary = {
    "dataset": "QUWI",
    "protocol": "writer_disjoint_exhaustive_verification",
    "development_train": {
        "writers": 226,
        "images": 904,
        "used_for_pair_evaluation": False,
    },
    "validation": {
        "writers": 56,
        "images": 224,
        "pairs": len(validation_pairs_df),
        "genuine_pairs": int(
            (
                validation_pairs_df["pair_label"] == 1
            ).sum()
        ),
        "impostor_pairs": int(
            (
                validation_pairs_df["pair_label"] == 0
            ).sum()
        ),
        "pairs_per_condition": 56**2,
    },
    "official_test": {
        "writers": 193,
        "images": 772,
        "pairs": len(official_test_pairs_df),
        "genuine_pairs": int(
            (
                official_test_pairs_df["pair_label"] == 1
            ).sum()
        ),
        "impostor_pairs": int(
            (
                official_test_pairs_df["pair_label"] == 0
            ).sum()
        ),
        "pairs_per_condition": 193**2,
    },
    "conditions": {
        "total": 6,
        "within_script": 2,
        "cross_script": 4,
        "definitions": pair_conditions_df.to_dict(
            orient="records"
        ),
    },
    "writer_leakage": {
        "development_validation_overlap": 0,
        "development_test_overlap": 0,
        "validation_test_overlap": 0,
    },
    "integrity": {
        "validation_label_integrity": bool(
            validation_label_integrity
        ),
        "official_test_label_integrity": bool(
            test_label_integrity
        ),
        "validation_pair_type_integrity": bool(
            validation_type_integrity
        ),
        "official_test_pair_type_integrity": bool(
            test_type_integrity
        ),
        "validation_duplicate_pairs": int(
            validation_pairs_df.duplicated(
                subset=[
                    "condition_id",
                    "filename_a",
                    "filename_b",
                ]
            ).sum()
        ),
        "official_test_duplicate_pairs": int(
            official_test_pairs_df.duplicated(
                subset=[
                    "condition_id",
                    "filename_a",
                    "filename_b",
                ]
            ).sum()
        ),
        "validation_same_file_pairs": int(
            (
                validation_pairs_df["filename_a"]
                == validation_pairs_df["filename_b"]
            ).sum()
        ),
        "official_test_same_file_pairs": int(
            (
                official_test_pairs_df["filename_a"]
                == official_test_pairs_df["filename_b"]
            ).sum()
        ),
    },
    "pair_balance": {
        "validation_genuine_pairs_per_writer": 1,
        "validation_impostor_pairs_per_writer_per_side": 55,
        "official_test_genuine_pairs_per_writer": 1,
        "official_test_impostor_pairs_per_writer_per_side": 192,
    },
    "generated_files": {
        "validation_pairs": str(
            VALIDATION_PAIR_PATH
        ),
        "official_test_pairs": str(
            TEST_PAIR_PATH
        ),
        "condition_definitions": str(
            CONDITION_PATH
        ),
        "condition_summary": str(
            CONDITION_SUMMARY_PATH
        ),
        "pair_balance_audit": str(
            PAIR_BALANCE_PATH
        ),
        "writer_leakage_audit": str(
            WRITER_LEAKAGE_PATH
        ),
        "writer_page_audit": str(
            PAGE_AUDIT_PATH
        ),
    },
}

with FINAL_SUMMARY_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        make_json_safe(
            final_protocol_summary
        ),
        file,
        indent=2,
    )

final_paths = [
    VALIDATION_PAIR_PATH,
    TEST_PAIR_PATH,
    CONDITION_PATH,
    CONDITION_SUMMARY_PATH,
    PAIR_BALANCE_PATH,
    WRITER_LEAKAGE_PATH,
    PAGE_AUDIT_PATH,
    PROTOCOL_SUMMARY_PATH,
    FINAL_SUMMARY_PATH,
]

for path in final_paths:
    print(
        path.name,
        ":",
        path.exists(),
    )

print(
    "\nValidation pairs:",
    len(validation_pairs_df),
)

print(
    "Official-test pairs:",
    len(official_test_pairs_df),
)

print(
    "Total verification pairs:",
    len(validation_pairs_df)
    + len(official_test_pairs_df),
)

print(
    "Writer-disjoint protocol:",
    (
        len(development_writers & validation_writers) == 0
        and len(development_writers & test_writers) == 0
        and len(validation_writers & test_writers) == 0
    ),
)

quwi_validation_verification_pairs.csv : True
quwi_official_test_verification_pairs.csv : True
quwi_verification_conditions.csv : True
pair_condition_summary.csv : True
pair_balance_audit.csv : True
writer_leakage_audit.json : True
writer_page_audit.csv : True
quwi_verification_protocol_summary.json : True
verification_protocol_final_summary.json : True

Validation pairs: 18816
Official-test pairs: 223494
Total verification pairs: 242310
Writer-disjoint protocol: True


## Verification protocol conclusion

A deterministic and exhaustive writer-disjoint verification protocol was constructed for QUWI.

The development-training writers are excluded from verification evaluation. The validation set contains 56 unseen writers and the official-test set contains 193 additional unseen writers, with zero writer overlap across all three experiment splits.

Six verification conditions were defined:

- Arabic variable-text versus Arabic same-text
- English variable-text versus English same-text
- Arabic variable-text versus English variable-text
- Arabic variable-text versus English same-text
- Arabic same-text versus English variable-text
- Arabic same-text versus English same-text

For every condition, all writers from page A were exhaustively paired with all writers from page B. Same-writer combinations form genuine pairs and different-writer combinations form impostor pairs.

The resulting protocol contains:

- 18,816 validation pairs
- 336 validation genuine pairs
- 18,480 validation impostor pairs
- 223,494 official-test pairs
- 1,158 official-test genuine pairs
- 222,336 official-test impostor pairs
- 242,310 verification pairs in total

Pair construction is balanced at the writer level. In each validation condition, every writer contributes one genuine pair and appears in 55 impostor pairs on each side of the comparison. In each official-test condition, every writer contributes one genuine pair and appears in 192 impostor pairs on each side.

All integrity checks passed:

- Zero writer overlap across development, validation, and official-test splits
- Complete coverage of all validation and official-test images
- Zero unexpected filenames
- Zero same-file verification pairs
- Zero duplicate condition-specific pairs
- Correct genuine and impostor labels
- Correct page and condition mappings
- Balanced impostor participation across writers

Because the exhaustive protocol contains substantially more impostor than genuine pairs, ordinary classification accuracy will not be used as the primary verification metric. Subsequent experiments will evaluate similarity scores using ROC-AUC, equal error rate, false-accept and false-reject operating points, and TAR at fixed FAR thresholds.

Threshold selection and model development will use the writer-disjoint validation writers. The official-test writers will remain reserved for final evaluation.

The next stage will extract writer embeddings from existing ResNet-18 baselines and evaluate genuine and impostor similarity scores on this fixed verification protocol.